# Attack failures, sample sizes and running times

* B.3 / B.4: how often did each attack fail (p* = search cap) and are these values part of the reported averages?
* B.17: sample sizes per dataset, network and experiment
* B.23: medians and quantiles of the running times next to the means

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from common import *

set_style()
pd.set_option("display.max_rows", 200)

In [ ]:
cifar_df = load_cifar()
cifar_df["dataset"] = "CIFAR-10"
cifar_df["failed"] = failure_mask(cifar_df, CIFAR_CAP)
imagenet_df = load_imagenet()
imagenet_df["dataset"] = "ImageNet"
imagenet_df["failed"] = failure_mask(imagenet_df, IMAGENET_CAP)
attack_df = pd.concat([cifar_df, imagenet_df])

## Attack failure rates (B.3, B.4)

In [ ]:
failure_df = attack_df.groupby(["dataset", "network", "verifier"]).agg(
    images=("image_id", "size"), failed=("failed", "sum"), failure_rate=("failed", "mean"))
failure_df

In [ ]:
print(save_table(failure_df, "rebuttal_attack_failure_rates"))

Average p* with and without the capped values:

In [ ]:
attack_df.groupby(["dataset", "verifier"]).apply(lambda d: pd.Series({
    "mean_incl_failures": d.smallest_sat_value.mean(),
    "mean_successful_only": d[~d.failed].smallest_sat_value.mean(),
    "median_incl_failures": d.smallest_sat_value.median()}))

## Sample sizes (B.17)

In [ ]:
sizes = attack_df.groupby(["dataset", "network", "verifier"]).image_id.agg(
    correctly_classified_evaluated="size", min_id="min", max_id="max")
sizes

## Running time distribution (B.23)

In [ ]:
time_df = attack_df.groupby(["dataset", "verifier"]).total_time.describe(percentiles=[.25, .5, .75, .95])
time_df